# 01 — Data collection

This notebook is for the CHIRPS (rainfall) / GloFAS (discharge) extraction and the
EM-DAT flood register merge, the same work that currently lives in your existing
`Historical_Data_extraction 1.ipynb` and `Historical_Data.ipynb` in Drive.

Paste your existing extraction and merge cells into the placeholder sections below.
Nothing here is invented for you, Earth Engine authentication and the exact
`CATCHMENTS` / `CATCHMENT_COORDS` dictionaries are yours to carry over unchanged.

**What this notebook adds on top of what you already have:** it clones the repo so
the result lands in `data/raw/` with a new version number, exactly where
`src/config.py` expects it, and pushes that file back to GitHub so it is one
`git pull` away in VS Code, instead of a manual download/upload.

In [ ]:
# --- One-time per Colab session: clone the repo and install dependencies ---
import os
from google.colab import userdata

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")  # set once: key icon in the left sidebar, see colab/README.md
REPO = "DaniellaWambua/Real-Time-Flood-Data-Pipeline"
BRANCH = "colab-training"
REPO_DIR = "/content/repo"

if not os.path.isdir(REPO_DIR):
    !git clone -b {BRANCH} https://{GITHUB_TOKEN}@github.com/{REPO}.git {REPO_DIR}
%cd {REPO_DIR}
!git pull

!pip install -q -r requirements-colab.txt


## Earth Engine + Open-Meteo extraction

Paste the cell(s) from `Historical_Data_extraction 1.ipynb` that pull CHIRPS via
Earth Engine (batch export to Drive) and GloFAS via the Open-Meteo REST loop, and
write `chirps_raw.csv` / `glofas_raw.csv`. If a step still needs to land in Drive
first because of Earth Engine's asynchronous export, mount Drive for that one step
only:

```python
from google.colab import drive
drive.mount("/content/drive")
```

In [ ]:
# TODO: paste your Earth Engine / Open-Meteo extraction cells here


## Merge, flood register and feature engineering

Paste the cells from `Historical_Data.ipynb` that merge the two raw files, apply
the EM-DAT-sourced flood register (see `training-results.md`, Phase 6, for the
17-event register and its sourcing), and engineer the feature columns into the
final merged dataframe.

In [ ]:
# TODO: paste your merge / flood-register / feature-engineering cells here
# End with a single dataframe called `final_df`, one row per catchment per day.


## Write the new dataset version into the repo and push it back

Bump `NEW_VERSION` each time you produce a genuinely new dataset (matching the
`Dataset_labels_v4.csv` ... `Dataset_labels_v7.csv` convention already in
`data/raw/`), so earlier versions stay available for comparison.

In [ ]:
NEW_VERSION = "v8"  # bump this each time you regenerate the dataset

out_path = f"data/raw/Dataset_labels_{NEW_VERSION}.csv"
final_df.to_csv(out_path, index=False)
print(f"Wrote {len(final_df)} rows to {out_path}")


In [ ]:
!git add data/raw/
!git -c user.email="167078857+DaniellaWambua@users.noreply.github.com" -c user.name="Daniella Wambua" \
     commit -m "Add Dataset_labels_{NEW_VERSION}.csv from Colab data collection"
!git push origin colab-training


## Back in VS Code

```bash
git checkout colab-training
git pull
```

Then point `src/config.py` at the new file (`RAW_FEATURES_LABELS_FILE`) the same
way `FLOOD_DATASET=v6` already switches between versions today, when you are
ready to train against it.